# Assessment plateforme et migration Fabric Runtime 1.2 → 1.3

## Objectif

1. Inventorier **tout le tenant**, et pas seulement les workspaces où je possède un rôle.
2. Détecter les **notebooks et Spark Job Definitions impactés** et identifier les équipes à prévenir.
3. Migrer les workspaces et Environment items encore en Runtime 1.2.

## Pourquoi la cellule d'inventaire ne voyait qu'une partie des workspaces

Le rôle **Fabric Administrator** ne donne pas accès au contenu des workspaces. Les deux familles d'API n'ont pas la même portée :

| API | Portée | Permission requise |
| --- | --- | --- |
| `GET /v1/admin/workspaces` | **Tenant** | Fabric Administrator, `Tenant.Read.All` |
| `GET /v1/admin/items?type=Notebook` | **Tenant** | Fabric Administrator, `Tenant.Read.All` |
| `GET /v1/workspaces/{id}/spark/settings` | **Workspace** | rôle *Viewer* minimum |
| `PATCH /v1/workspaces/{id}/spark/settings` | **Workspace** | rôle *Admin* |
| `GET`/`PATCH` `.../environments/{id}/staging/sparkcompute` | **Item** | lecture / écriture sur l'Environment |

`fabric.list_workspaces()` s'appuie sur la portée utilisateur : il ne retourne que les workspaces où vous êtes Admin, Membre, Contributeur ou Viewer. La **découverte** est donc tenant-wide, mais la **lecture du runtime et la migration ne le sont pas**.

Ce notebook :

- utilise les **Admin APIs** pour l'inventaire complet du tenant ;
- marque explicitement chaque workspace **inaccessible** au lieu de le masquer silencieusement ;
- propose une **élévation just-in-time** du rôle Workspace Admin pour la phase d'écriture.

Pour couvrir l'ensemble de la plateforme en écriture, trois options :

1. `AUTO_ELEVATE = True` — le notebook s'attribue le rôle Admin workspace par workspace, migre, puis le retire.
2. Ajouter un **groupe de sécurité** « Plateforme Data » comme Admin sur les workspaces concernés.
3. Portail d'administration → **Workspaces** → *Access*, pour un traitement ponctuel.

## Attention sur la cible 1.3

Runtime 1.3 (Spark 3.5) quitte le statut GA le **30 septembre 2026**, puis passe en Long Term Support **jusqu'à mars 2027**. C'est une cible **transitoire** : elle limite le risque de compatibilité (Spark 3.4 → 3.5, Python 3.10 → 3.11) mais impose une seconde migration vers Runtime 2.0 (Spark 4.1, support jusqu'en août 2028) d'ici mars 2027.

Passer `TARGET_RUNTIME` à `"2.0"` évite ce double effort si les dépendances le permettent.

## Limites des Admin APIs

- **200 requêtes par heure**, d'où le garde-fou `MAX_ADMIN_CALLS`.
- Pagination par `continuationToken`, 10 000 enregistrements par page.
- `GET /v1/admin/items` est en **préversion**.


# 0. Paramètres et utilitaires


In [ ]:
import time
import json
import base64
from urllib.parse import urlencode

import pandas as pd
import sempy.fabric as fabric

# ============================================================
# PARAMETRES
# ============================================================

TARGET_RUNTIME  = "1.3"        # "2.0" pour viser directement le runtime GA le plus recent
SOURCE_RUNTIMES = {"1.2"}      # runtimes consideres comme a migrer

USE_ADMIN_API    = True        # inventaire tenant-wide (role Fabric Administrator requis)
INCLUDE_PERSONAL = False       # inclure les "My workspaces"

# Deux garde-fous cumulatifs : la portee etant tenant-wide, un seul booleen ne suffit pas.
DRY_RUN           = True       # True = aucune ecriture, on ne produit que le plan
CONFIRMATION      = ""         # doit valoir exactement f"UPGRADE TO {TARGET_RUNTIME}"
TARGET_WORKSPACES = []         # [] = tous les workspaces detectes en SOURCE_RUNTIMES

AUTO_ELEVATE           = False      # s'attribuer le role Workspace Admin le temps de la migration
ELEVATE_PRINCIPAL      = None       # UPN (User) ou objectId (App / Group)
ELEVATE_PRINCIPAL_TYPE = "User"     # "User" | "App" | "Group"
REVOKE_AFTER_MIGRATION = True       # retirer le role apres migration

RESOLVE_CONTACTS    = True     # resoudre les equipes a prevenir (consomme du quota admin)
DEEP_SCAN_NOTEBOOKS = False    # lire la definition des notebooks pour trouver l'Environment attache

MAX_ADMIN_CALLS = 180          # garde-fou sous la limite Microsoft de 200 appels/heure

EXPECTED_CONFIRMATION = f"UPGRADE TO {TARGET_RUNTIME}"
WRITE_ENABLED = (not DRY_RUN) and CONFIRMATION == EXPECTED_CONFIRMATION

client = fabric.FabricRestClient()

_admin_calls = 0

# ============================================================
# UTILITAIRES HTTP
# ============================================================

def _url(path, params=None):
    params = {k: v for k, v in (params or {}).items() if v is not None}
    return f"{path}?{urlencode(params)}" if params else path


def api_call(method, path, params=None, json_body=None, max_retries=4, is_admin=False):
    """Appel REST avec gestion du throttling 429 et garde-fou sur le quota des API admin."""
    global _admin_calls

    if is_admin:
        if _admin_calls >= MAX_ADMIN_CALLS:
            raise RuntimeError(
                f"Garde-fou atteint : {_admin_calls} appels admin. "
                "La limite Microsoft est de 200/heure. Relancer plus tard ou ajuster MAX_ADMIN_CALLS."
            )
        _admin_calls += 1

    url = _url(path, params)
    fn = getattr(client, method)

    for attempt in range(max_retries + 1):
        response = fn(url, json=json_body) if json_body is not None else fn(url)

        if response.status_code == 429 and attempt < max_retries:
            wait = int(response.headers.get("Retry-After", 30))
            print(f"  429 throttling, pause {wait}s ({path})")
            time.sleep(min(wait, 300))
            continue

        return response

    return response


def _extract_list(body):
    """Les API Fabric n'exposent pas toutes leur collection sous la meme cle."""
    for key in ("value", "workspaces", "itemEntities", "items", "accessDetails", "data"):
        if isinstance(body.get(key), list):
            return body[key]

    for value in body.values():
        if isinstance(value, list):
            return value

    return []


def get_paginated(path, params=None, is_admin=False, max_pages=100):
    """Suit continuationToken jusqu'a epuisement de la collection."""
    collected, token, pages = [], None, 0

    while pages < max_pages:
        query = dict(params or {})

        if token:
            query["continuationToken"] = token

        response = api_call("get", path, params=query, is_admin=is_admin)

        if response.status_code != 200:
            raise RuntimeError(f"{path} -> {response.status_code} : {response.text[:400]}")

        body = response.json()
        collected.extend(_extract_list(body))

        token = body.get("continuationToken")
        pages += 1

        if not token:
            break

    return collected


def admin_list_items(item_type):
    """Liste tenant-wide d'un type d'item, avec repli si le filtre serveur est refuse."""
    try:
        return get_paginated("/v1/admin/items", {"type": item_type}, is_admin=True)

    except RuntimeError as ex:
        print(f"  Filtre type={item_type} refuse ({ex}). Repli sur un listing complet.")
        every_item = get_paginated("/v1/admin/items", is_admin=True)
        return [i for i in every_item if str(i.get("type", "")).lower() == item_type.lower()]


def wait_for_operation(response, timeout_s=1800, label=""):
    """Suit une operation longue jusqu'a son etat terminal. Un 202 ne prouve pas le succes."""
    if response.status_code == 200:
        return "Succeeded", None

    if response.status_code != 202:
        return "Failed", f"HTTP {response.status_code}: {response.text[:300]}"

    operation_id = response.headers.get("x-ms-operation-id")
    delay = int(response.headers.get("Retry-After", 20))

    if not operation_id:
        return "Unknown", "202 sans en-tete x-ms-operation-id"

    deadline = time.time() + timeout_s

    while time.time() < deadline:
        time.sleep(min(delay, 60))

        poll = api_call("get", f"/v1/operations/{operation_id}")

        if poll.status_code != 200:
            return "Unknown", f"Polling HTTP {poll.status_code}"

        body = poll.json()
        status = body.get("status", "Unknown")

        if status in ("Succeeded", "Failed", "Cancelled", "Undefined"):
            error = json.dumps(body.get("error")) if body.get("error") else None
            return status, error

    return "Timeout", f"Non termine apres {timeout_s}s ({label})"


print(f"Cible         : Runtime {TARGET_RUNTIME}")
print(f"Sources       : {sorted(SOURCE_RUNTIMES)}")
print(f"Admin API     : {USE_ADMIN_API}")
print(f"DRY_RUN       : {DRY_RUN}")
print(f"Ecriture      : {'ACTIVEE' if WRITE_ENABLED else 'BLOQUEE'}")

if not WRITE_ENABLED and not DRY_RUN:
    print(f"  CONFIRMATION attendue : \"{EXPECTED_CONFIRMATION}\"")


# 1. Inventory

## 1.a. Workspaces

In [ ]:
# ============================================================
# 1.a. Workspaces : perimetre tenant + runtime par defaut
# ============================================================

# Portee utilisateur : sert de reference pour mesurer l'angle mort.
user_workspaces = fabric.list_workspaces()
user_workspace_ids = set(user_workspaces["Id"].astype(str))

print(f"Workspaces visibles via mon role utilisateur : {len(user_workspace_ids)}")

if USE_ADMIN_API:
    raw_workspaces = get_paginated("/v1/admin/workspaces", is_admin=True)

    workspaces_df = pd.DataFrame([
        {
            "WorkspaceId":   str(ws.get("id")),
            "WorkspaceName": ws.get("name") or ws.get("displayName"),
            "Type":          ws.get("type"),
            "State":         ws.get("state"),
            "CapacityId":    ws.get("capacityId"),
        }
        for ws in raw_workspaces
    ])

    print(f"Workspaces retournes par l'Admin API         : {len(workspaces_df)}")

else:
    workspaces_df = pd.DataFrame({
        "WorkspaceId":   user_workspaces["Id"].astype(str),
        "WorkspaceName": user_workspaces["Name"],
        "Type":          "Workspace",
        "State":         "Active",
        "CapacityId":    None,
    })

# Filtrage cote client : la casse des valeurs type/state varie selon les endpoints.
if len(workspaces_df):
    workspaces_df = workspaces_df[
        workspaces_df["State"].astype(str).str.lower() == "active"
    ]

    if not INCLUDE_PERSONAL:
        workspaces_df = workspaces_df[
            workspaces_df["Type"].astype(str).str.lower() != "personal"
        ]

    workspaces_df["HasMyAccess"] = workspaces_df["WorkspaceId"].isin(user_workspace_ids)

workspaces_df = workspaces_df.reset_index(drop=True)

print(f"Workspaces actifs retenus                    : {len(workspaces_df)}")

# ------------------------------------------------------------
# Lecture du runtime par defaut.
# API workspace-scoped : le role Fabric Administrator ne suffit pas,
# il faut au minimum le role Viewer sur chaque workspace.
# ------------------------------------------------------------

settings_rows = []

for _, ws in workspaces_df.iterrows():

    workspace_id = ws["WorkspaceId"]

    row = {
        "WorkspaceId":        workspace_id,
        "WorkspaceName":      ws["WorkspaceName"],
        "CapacityId":         ws["CapacityId"],
        "HasMyAccess":        ws["HasMyAccess"],
        "RuntimeVersion":     None,
        "DefaultEnvironment": None,
        "AccessStatus":       None,
        "Detail":             None,
    }

    try:
        response = api_call("get", f"/v1/workspaces/{workspace_id}/spark/settings")

        if response.status_code == 200:
            environment = response.json().get("environment") or {}
            row["RuntimeVersion"]     = environment.get("runtimeVersion")
            row["DefaultEnvironment"] = environment.get("name") or None
            row["AccessStatus"]       = "Ok"

        elif response.status_code in (401, 403):
            row["AccessStatus"] = "Forbidden"
            row["Detail"] = "Aucun role sur ce workspace : runtime illisible et migration impossible"

        elif response.status_code == 404:
            row["AccessStatus"] = "NotFound"

        else:
            row["AccessStatus"] = f"HTTP{response.status_code}"
            row["Detail"] = response.text[:200]

    except Exception as ex:
        row["AccessStatus"] = "Exception"
        row["Detail"] = str(ex)[:200]

    settings_rows.append(row)

workspace_runtime_df = pd.DataFrame(settings_rows)

# ------------------------------------------------------------
# Synthese : rien n'est masque, chaque workspace a un statut.
# ------------------------------------------------------------

runtime12_workspaces = workspace_runtime_df[
    workspace_runtime_df["RuntimeVersion"].isin(SOURCE_RUNTIMES)
]

unreadable_workspaces = workspace_runtime_df[
    workspace_runtime_df["AccessStatus"] != "Ok"
]

print("\nRepartition des statuts d'acces :")
print(workspace_runtime_df["AccessStatus"].value_counts().to_string())

print("\nRepartition des runtimes lisibles :")
print(workspace_runtime_df["RuntimeVersion"].value_counts(dropna=False).to_string())

print(f"\nWorkspaces en {sorted(SOURCE_RUNTIMES)} : {len(runtime12_workspaces)}")
print(f"Workspaces non lisibles (angle mort)  : {len(unreadable_workspaces)}")

if len(unreadable_workspaces):
    print(
        "\nCes workspaces necessitent un role avant migration : "
        "AUTO_ELEVATE, groupe de securite, ou portail d'administration."
    )
    display(unreadable_workspaces[["WorkspaceName", "AccessStatus", "Detail"]])

display(runtime12_workspaces)


## 1.b. Environments

In [ ]:
# ============================================================
# 1.b. Environments : decouverte tenant-wide + runtime effectif
# ============================================================

workspace_names = dict(
    zip(workspace_runtime_df["WorkspaceId"], workspace_runtime_df["WorkspaceName"])
)

known_workspace_ids = set(workspace_runtime_df["WorkspaceId"])

if USE_ADMIN_API:
    # Un seul balayage tenant-wide remplace N appels par workspace.
    raw_environments = admin_list_items("Environment")

    environment_items = [
        {
            "WorkspaceId":     str(item.get("workspaceId")),
            "EnvironmentId":   str(item.get("id")),
            "EnvironmentName": item.get("displayName") or item.get("name"),
        }
        for item in raw_environments
        if str(item.get("workspaceId")) in known_workspace_ids
    ]

else:
    environment_items = []

    for workspace_id in workspace_runtime_df["WorkspaceId"]:
        response = api_call("get", f"/v1/workspaces/{workspace_id}/environments")

        if response.status_code != 200:
            continue

        for item in _extract_list(response.json()):
            environment_items.append({
                "WorkspaceId":     workspace_id,
                "EnvironmentId":   str(item.get("id")),
                "EnvironmentName": item.get("displayName"),
            })

print(f"Environment items decouverts : {len(environment_items)}")

# ------------------------------------------------------------
# Lecture des configurations Spark.
# "published" = ce qui s'applique reellement aux sessions.
# "staging"   = ce qui est en attente de publication.
# ------------------------------------------------------------

environment_rows = []

for item in environment_items:

    workspace_id   = item["WorkspaceId"]
    environment_id = item["EnvironmentId"]

    row = {
        "WorkspaceName":    workspace_names.get(workspace_id),
        "WorkspaceId":      workspace_id,
        "EnvironmentName":  item["EnvironmentName"],
        "EnvironmentId":    environment_id,
        "PublishedRuntime": None,
        "StagingRuntime":   None,
        "AccessStatus":     None,
        "Detail":           None,
    }

    base = f"/v1/workspaces/{workspace_id}/environments/{environment_id}"

    try:
        published = api_call("get", f"{base}/sparkcompute", params={"beta": "false"})

        if published.status_code == 200:
            row["PublishedRuntime"] = published.json().get("runtimeVersion")
            row["AccessStatus"] = "Ok"

        elif published.status_code in (401, 403):
            row["AccessStatus"] = "Forbidden"
            row["Detail"] = "Pas de permission de lecture sur cet Environment"

        else:
            row["AccessStatus"] = f"HTTP{published.status_code}"
            row["Detail"] = published.text[:200]

        if row["AccessStatus"] == "Ok":
            staging = api_call("get", f"{base}/staging/sparkcompute", params={"beta": "false"})

            if staging.status_code == 200:
                row["StagingRuntime"] = staging.json().get("runtimeVersion")

    except Exception as ex:
        row["AccessStatus"] = "Exception"
        row["Detail"] = str(ex)[:200]

    environment_rows.append(row)

environment_df = pd.DataFrame(environment_rows)

# Le runtime effectif est celui publie ; le staging sert a detecter une modification non publiee.
if len(environment_df):
    environment_df["EffectiveRuntime"] = environment_df["PublishedRuntime"].fillna(
        environment_df["StagingRuntime"]
    )

    environment_df["PendingPublish"] = (
        environment_df["StagingRuntime"].notna()
        & (environment_df["StagingRuntime"] != environment_df["PublishedRuntime"])
    )

    runtime12_environments = environment_df[
        environment_df["EffectiveRuntime"].isin(SOURCE_RUNTIMES)
    ]

    unreadable_environments = environment_df[environment_df["AccessStatus"] != "Ok"]

else:
    environment_df = pd.DataFrame(
        columns=[
            "WorkspaceName", "WorkspaceId", "EnvironmentName", "EnvironmentId",
            "PublishedRuntime", "StagingRuntime", "AccessStatus", "Detail",
            "EffectiveRuntime", "PendingPublish",
        ]
    )
    runtime12_environments = environment_df
    unreadable_environments = environment_df

print(f"Environments en {sorted(SOURCE_RUNTIMES)} : {len(runtime12_environments)}")
print(f"Environments non lisibles              : {len(unreadable_environments)}")

if len(environment_df):
    print("\nRepartition des runtimes effectifs :")
    print(environment_df["EffectiveRuntime"].value_counts(dropna=False).to_string())

display(runtime12_environments)


## 1.c. Notebooks et Spark Job Definitions impactés

Détecte tous les notebooks et Spark Job Definitions du tenant via `GET /v1/admin/items`, puis détermine le runtime qui s'appliquera réellement à chacun.

Règle de résolution, du plus prioritaire au moins prioritaire :

1. **Environment attaché à l'item** — écrase le runtime du workspace. Nécessite `DEEP_SCAN_NOTEBOOKS = True`, car cette information vit dans la définition du notebook et non dans la liste des items.
2. **Runtime par défaut du workspace** — s'applique aux items sans Environment.

Statuts produits :

| Impact | Signification |
| --- | --- |
| `Impacte` | Le runtime effectif est dans `SOURCE_RUNTIMES` : l'item changera de runtime. |
| `A verifier` | Le workspace est à jour mais contient un Environment encore en 1.2. |
| `Indetermine` | Workspace non lisible : impact inconnu tant que l'accès n'est pas obtenu. |
| `Non impacte` | Déjà sur un runtime hors périmètre. |


In [ ]:
import re

# ============================================================
# 1.c. Notebooks et Spark Job Definitions impactes
# ============================================================

SCANNED_TYPES = ["Notebook", "SparkJobDefinition"]

item_rows = []

if USE_ADMIN_API:
    for item_type in SCANNED_TYPES:
        for item in admin_list_items(item_type):

            workspace_id = str(item.get("workspaceId"))

            if workspace_id not in known_workspace_ids:
                continue

            item_rows.append({
                "WorkspaceId":   workspace_id,
                "WorkspaceName": workspace_names.get(workspace_id),
                "ItemId":        str(item.get("id")),
                "ItemName":      item.get("displayName") or item.get("name"),
                "Type":          item_type,
            })

else:
    for workspace_id in workspace_runtime_df["WorkspaceId"]:
        for item_type in SCANNED_TYPES:

            response = api_call(
                "get",
                f"/v1/workspaces/{workspace_id}/items",
                params={"type": item_type},
            )

            if response.status_code != 200:
                continue

            for item in _extract_list(response.json()):
                item_rows.append({
                    "WorkspaceId":   workspace_id,
                    "WorkspaceName": workspace_names.get(workspace_id),
                    "ItemId":        str(item.get("id")),
                    "ItemName":      item.get("displayName"),
                    "Type":          item_type,
                })

items_df = pd.DataFrame(item_rows)

print(f"Items Spark decouverts : {len(items_df)}")

if len(items_df):
    print(items_df["Type"].value_counts().to_string())

# ------------------------------------------------------------
# Scan optionnel des definitions : identifie l'Environment reellement
# attache a un notebook, information absente de la liste des items.
# ------------------------------------------------------------

def find_attached_environment(workspace_id, notebook_id):
    """Best effort : renvoie l'environmentId trouve dans la definition du notebook."""
    response = api_call(
        "post",
        f"/v1/workspaces/{workspace_id}/notebooks/{notebook_id}/getDefinition",
    )

    if response.status_code == 202:
        status, _ = wait_for_operation(response, timeout_s=300, label="getDefinition")

        if status != "Succeeded":
            return None

        operation_id = response.headers.get("x-ms-operation-id")
        response = api_call("get", f"/v1/operations/{operation_id}/result")

    if response.status_code != 200:
        return None

    definition = (response.json() or {}).get("definition") or {}

    for part in definition.get("parts", []):

        payload = part.get("payload")

        if not payload:
            continue

        try:
            text = base64.b64decode(payload).decode("utf-8", errors="ignore")
        except Exception:
            continue

        match = re.search(r'"environmentId"\s*:\s*"([0-9a-fA-F-]{36})"', text)

        if match:
            return match.group(1)

    return None


if len(items_df):
    items_df["AttachedEnvironmentId"] = None

if DEEP_SCAN_NOTEBOOKS and len(items_df):

    notebooks = items_df[items_df["Type"] == "Notebook"]
    print(f"\nDeep scan de {len(notebooks)} notebooks (necessite un role sur le workspace)...")

    for index, item in notebooks.iterrows():
        try:
            items_df.at[index, "AttachedEnvironmentId"] = find_attached_environment(
                item["WorkspaceId"], item["ItemId"]
            )
        except Exception:
            pass

# ------------------------------------------------------------
# Classification de l'impact
# ------------------------------------------------------------

environment_runtime = {}

if len(environment_df):
    environment_runtime = dict(
        zip(environment_df["EnvironmentId"], environment_df["EffectiveRuntime"])
    )

workspace_runtime = dict(
    zip(workspace_runtime_df["WorkspaceId"], workspace_runtime_df["RuntimeVersion"])
)

workspace_access = dict(
    zip(workspace_runtime_df["WorkspaceId"], workspace_runtime_df["AccessStatus"])
)

environments_by_workspace = {}

if len(environment_df):
    for workspace_id, group in environment_df.groupby("WorkspaceId"):
        environments_by_workspace[workspace_id] = set(group["EffectiveRuntime"].dropna())


def classify(row):
    workspace_id = row["WorkspaceId"]
    attached = row.get("AttachedEnvironmentId")

    # Un Environment attache prime sur le runtime par defaut du workspace.
    if attached and attached in environment_runtime:
        runtime = environment_runtime[attached]
        if runtime in SOURCE_RUNTIMES:
            return pd.Series([runtime, "Environment attache", "Impacte"])
        return pd.Series([runtime, "Environment attache", "Non impacte"])

    runtime = workspace_runtime.get(workspace_id)

    if runtime in SOURCE_RUNTIMES:
        return pd.Series([runtime, "Runtime par defaut du workspace", "Impacte"])

    if workspace_access.get(workspace_id) != "Ok":
        return pd.Series([None, "Workspace non lisible", "Indetermine"])

    # Le workspace est a jour, mais un Environment du workspace peut rester en 1.2.
    if SOURCE_RUNTIMES & environments_by_workspace.get(workspace_id, set()):
        return pd.Series([runtime, "Environment du workspace en source", "A verifier"])

    return pd.Series([runtime, "Runtime par defaut du workspace", "Non impacte"])


if len(items_df):
    items_df[["EffectiveRuntime", "RuntimeSource", "Impact"]] = items_df.apply(classify, axis=1)

    impacted_items_df = items_df[items_df["Impact"].isin(["Impacte", "A verifier"])]

    print("\nClassification des items :")
    print(items_df["Impact"].value_counts().to_string())

    print("\nTop workspaces impactes :")
    print(
        impacted_items_df.groupby("WorkspaceName").size()
        .sort_values(ascending=False).head(20).to_string()
    )

    display(impacted_items_df.head(100))

else:
    items_df = pd.DataFrame(
        columns=[
            "WorkspaceId", "WorkspaceName", "ItemId", "ItemName", "Type",
            "AttachedEnvironmentId", "EffectiveRuntime", "RuntimeSource", "Impact",
        ]
    )
    impacted_items_df = items_df

# Perimetre de communication et de migration : workspaces porteurs d'un impact reel.
impacted_workspace_ids = (
    set(impacted_items_df["WorkspaceId"])
    | set(runtime12_workspaces["WorkspaceId"])
    | set(runtime12_environments["WorkspaceId"])
)

print(f"\nWorkspaces impactes au total : {len(impacted_workspace_ids)}")


## 1.d. Équipes à prévenir

Résout les Admins et Membres de chaque workspace impacté via `GET /v1/admin/workspaces/{id}/users`, puis produit une table de notification prête à être exportée.

Cette étape consomme **un appel admin par workspace impacté** : elle est donc limitée aux workspaces réellement concernés, et non à l'ensemble du tenant.


In [ ]:
# ============================================================
# 1.d. Equipes a prevenir
# ============================================================

contacts_df = pd.DataFrame()
notification_df = pd.DataFrame()

if not RESOLVE_CONTACTS:
    print("RESOLVE_CONTACTS = False : etape ignoree.")

elif not impacted_workspace_ids:
    print("Aucun workspace impacte : rien a notifier.")

else:
    target_ids = sorted(impacted_workspace_ids)

    print(f"Resolution des contacts sur {len(target_ids)} workspaces impactes.")
    print(f"Cout : {len(target_ids)} appels admin (quota consomme : {_admin_calls}/{MAX_ADMIN_CALLS}).")

    contact_rows = []

    for workspace_id in target_ids:

        try:
            response = api_call(
                "get",
                f"/v1/admin/workspaces/{workspace_id}/users",
                is_admin=True,
            )

            if response.status_code != 200:
                contact_rows.append({
                    "WorkspaceId":   workspace_id,
                    "WorkspaceName": workspace_names.get(workspace_id),
                    "Role":          None,
                    "DisplayName":   None,
                    "Principal":     None,
                    "PrincipalType": None,
                    "Status":        f"HTTP{response.status_code}",
                })
                continue

            for entry in _extract_list(response.json()):

                principal = entry.get("principal") or {}
                user_details = principal.get("userDetails") or {}
                spn_details = principal.get("servicePrincipalDetails") or {}

                access = entry.get("workspaceAccessDetails") or {}

                contact_rows.append({
                    "WorkspaceId":   workspace_id,
                    "WorkspaceName": workspace_names.get(workspace_id),
                    "Role":          access.get("role") or entry.get("role"),
                    "DisplayName":   principal.get("displayName"),
                    "Principal": (
                        user_details.get("userPrincipalName")
                        or spn_details.get("aadAppId")
                        or principal.get("id")
                    ),
                    "PrincipalType": principal.get("type"),
                    "Status":        "Ok",
                })

        except Exception as ex:
            contact_rows.append({
                "WorkspaceId":   workspace_id,
                "WorkspaceName": workspace_names.get(workspace_id),
                "Role":          None,
                "DisplayName":   None,
                "Principal":     None,
                "PrincipalType": None,
                "Status":        f"Exception: {str(ex)[:120]}",
            })

    contacts_df = pd.DataFrame(contact_rows)

    # ------------------------------------------------------------
    # Table de notification : une ligne par workspace impacte.
    # ------------------------------------------------------------

    notification_df = (
        workspace_runtime_df[workspace_runtime_df["WorkspaceId"].isin(impacted_workspace_ids)]
        [["WorkspaceId", "WorkspaceName", "RuntimeVersion", "AccessStatus"]]
        .copy()
    )

    owners = contacts_df[contacts_df["Role"].isin(["Admin", "Member"])]

    if len(owners):
        owners_by_workspace = (
            owners.groupby("WorkspaceId")["Principal"]
            .apply(lambda values: "; ".join(sorted({v for v in values if v})))
            .reset_index(name="Destinataires")
        )

        notification_df = notification_df.merge(
            owners_by_workspace, on="WorkspaceId", how="left"
        )
    else:
        notification_df["Destinataires"] = None

    if len(impacted_items_df):
        impact_counts = (
            impacted_items_df.groupby(["WorkspaceId", "Type"])
            .size()
            .unstack(fill_value=0)
            .reset_index()
        )

        notification_df = notification_df.merge(
            impact_counts, on="WorkspaceId", how="left"
        )

    notification_df["TargetRuntime"] = TARGET_RUNTIME
    notification_df["Destinataires"] = notification_df["Destinataires"].fillna(
        "AUCUN ADMIN IDENTIFIE"
    )

    orphans = int((notification_df["Destinataires"] == "AUCUN ADMIN IDENTIFIE").sum())

    print(f"\nWorkspaces a notifier       : {len(notification_df)}")
    print(f"Sans destinataire identifie : {orphans}")

    if orphans:
        print("Ces workspaces sont orphelins : traiter la propriete avant de migrer.")

    display(notification_df)


# 2. Migration

In [ ]:
# ============================================================
# 2. Migration vers TARGET_RUNTIME
#
# Sequence : Environments d'abord (portee reduite), workspace ensuite.
# Un 202 ne vaut pas succes : chaque publication est suivie jusqu'a
# son etat terminal, puis relue.
# ============================================================

operations = []
snapshots = []


def log_operation(scope, name, workspace, operation, status, detail=None):
    operations.append({
        "Scope":         scope,
        "Name":          name,
        "WorkspaceName": workspace,
        "Operation":     operation,
        "Status":        status,
        "Detail":        detail,
    })


# ------------------------------------------------------------
# Perimetre
# ------------------------------------------------------------

if TARGET_WORKSPACES:
    scope_df = workspace_runtime_df[
        workspace_runtime_df["WorkspaceName"].isin(TARGET_WORKSPACES)
    ]
else:
    scope_df = workspace_runtime_df[
        workspace_runtime_df["WorkspaceId"].isin(impacted_workspace_ids)
    ]

scope_ids = set(scope_df["WorkspaceId"])

environments_to_migrate = runtime12_environments[
    runtime12_environments["WorkspaceId"].isin(scope_ids)
    & (runtime12_environments["AccessStatus"] == "Ok")
]

workspaces_to_migrate = scope_df[
    scope_df["RuntimeVersion"].isin(SOURCE_RUNTIMES)
    & (scope_df["AccessStatus"] == "Ok")
]

blocked = scope_df[scope_df["AccessStatus"] != "Ok"]

print("=" * 70)
print("PLAN DE MIGRATION")
print("=" * 70)
print(f"Runtime cible                 : {TARGET_RUNTIME}")
print(f"Workspaces dans le perimetre  : {len(scope_df)}")
print(f"Environments a migrer         : {len(environments_to_migrate)}")
print(f"Workspaces a migrer           : {len(workspaces_to_migrate)}")
print(f"Bloques faute d'acces         : {len(blocked)}")
print(f"Items Spark concernes         : {len(impacted_items_df)}")

if len(blocked):
    print("\nWorkspaces bloques (role manquant) :")
    display(blocked[["WorkspaceName", "AccessStatus"]])

if not WRITE_ENABLED:
    print("\nMode plan : aucune ecriture effectuee.")
    print("Pour executer reellement la migration :")
    print("  1. verifier le plan ci-dessus ;")
    print("  2. prevenir les equipes listees dans notification_df ;")
    print("  3. restreindre TARGET_WORKSPACES a un workspace de DEV ;")
    print(f"  4. poser DRY_RUN = False et CONFIRMATION = \"{EXPECTED_CONFIRMATION}\".")

else:
    # --------------------------------------------------------
    # Elevation just-in-time
    # --------------------------------------------------------

    fabric_admin = None

    if AUTO_ELEVATE:
        try:
            import sempy.fabric.admin as fabric_admin
        except ImportError:
            print("sempy.fabric.admin indisponible : elevation impossible, mettre a jour semantic-link.")

        if fabric_admin and not ELEVATE_PRINCIPAL:
            print("ELEVATE_PRINCIPAL non renseigne : elevation desactivee.")
            fabric_admin = None

    def elevate(workspace_id, workspace_name):
        if not fabric_admin:
            return False

        try:
            fabric_admin.add_user_to_workspace(
                user=ELEVATE_PRINCIPAL,
                role="Admin",
                principal_type=ELEVATE_PRINCIPAL_TYPE,
                workspace=workspace_id,
            )
            log_operation("Workspace", workspace_name, workspace_name, "Elevate", "Succeeded")
            return True

        except Exception as ex:
            log_operation(
                "Workspace", workspace_name, workspace_name,
                "Elevate", "Failed", str(ex)[:300],
            )
            return False

    def revoke(workspace_id, workspace_name):
        remove = getattr(fabric_admin, "delete_user_from_workspace", None) if fabric_admin else None

        if not remove:
            log_operation(
                "Workspace", workspace_name, workspace_name,
                "Revoke", "Manual", "Retrait a effectuer manuellement",
            )
            return

        try:
            remove(user=ELEVATE_PRINCIPAL, workspace=workspace_id)
            log_operation("Workspace", workspace_name, workspace_name, "Revoke", "Succeeded")

        except Exception as ex:
            log_operation(
                "Workspace", workspace_name, workspace_name,
                "Revoke", "Failed", str(ex)[:300],
            )

    # --------------------------------------------------------
    # Environments
    # --------------------------------------------------------

    for _, env in environments_to_migrate.iterrows():

        workspace_id   = env["WorkspaceId"]
        environment_id = env["EnvironmentId"]
        env_name       = env["EnvironmentName"]
        ws_name        = env["WorkspaceName"]

        base = f"/v1/workspaces/{workspace_id}/environments/{environment_id}"

        elevated = elevate(workspace_id, ws_name) if AUTO_ELEVATE else False

        try:
            # Snapshot avant modification, pour le rollback.
            current = api_call("get", f"{base}/staging/sparkcompute", params={"beta": "false"})

            if current.status_code == 200:
                snapshots.append({
                    "Scope":         "Environment",
                    "WorkspaceName": ws_name,
                    "Name":          env_name,
                    "Config":        json.dumps(current.json()),
                })

            # Payload minimal : PATCH fusionne, le pool et les proprietes Spark sont preserves.
            update = api_call(
                "patch",
                f"{base}/staging/sparkcompute",
                params={"beta": "false"},
                json_body={"runtimeVersion": TARGET_RUNTIME},
            )

            if update.status_code not in (200, 202):
                log_operation(
                    "Environment", env_name, ws_name, "UpdateRuntime",
                    f"HTTP{update.status_code}", update.text[:300],
                )
                continue

            staged = api_call("get", f"{base}/staging/sparkcompute", params={"beta": "false"})
            staged_runtime = staged.json().get("runtimeVersion") if staged.status_code == 200 else None

            if staged_runtime != TARGET_RUNTIME:
                log_operation(
                    "Environment", env_name, ws_name, "UpdateRuntime",
                    "Failed", f"Staging = {staged_runtime}",
                )
                continue

            log_operation("Environment", env_name, ws_name, "UpdateRuntime", "Succeeded")

            # Publication : operation longue.
            publish = api_call("post", f"{base}/staging/publish", params={"beta": "false"})
            status, error = wait_for_operation(publish, label=f"publish {env_name}")

            if status != "Succeeded":
                log_operation("Environment", env_name, ws_name, "Publish", status, error)
                continue

            # Relecture de la configuration effectivement publiee.
            published = api_call("get", f"{base}/sparkcompute", params={"beta": "false"})
            published_runtime = published.json().get("runtimeVersion") if published.status_code == 200 else None

            log_operation(
                "Environment", env_name, ws_name, "Publish",
                "Succeeded" if published_runtime == TARGET_RUNTIME else "Failed",
                f"Runtime publie = {published_runtime}",
            )

        except Exception as ex:
            log_operation("Environment", env_name, ws_name, "Migration", "Exception", str(ex)[:300])

        finally:
            if elevated and REVOKE_AFTER_MIGRATION:
                revoke(workspace_id, ws_name)

    # --------------------------------------------------------
    # Workspaces
    # --------------------------------------------------------

    for _, ws in workspaces_to_migrate.iterrows():

        workspace_id = ws["WorkspaceId"]
        ws_name      = ws["WorkspaceName"]

        elevated = elevate(workspace_id, ws_name) if AUTO_ELEVATE else False

        try:
            snapshots.append({
                "Scope":         "Workspace",
                "WorkspaceName": ws_name,
                "Name":          ws_name,
                "Config":        json.dumps({"runtimeVersion": ws["RuntimeVersion"]}),
            })

            update = api_call(
                "patch",
                f"/v1/workspaces/{workspace_id}/spark/settings",
                json_body={"environment": {"runtimeVersion": TARGET_RUNTIME}},
            )

            if update.status_code not in (200, 202):
                log_operation(
                    "Workspace", ws_name, ws_name, "UpdateRuntime",
                    f"HTTP{update.status_code}", update.text[:300],
                )
                continue

            check = api_call("get", f"/v1/workspaces/{workspace_id}/spark/settings")
            observed = (check.json().get("environment") or {}).get("runtimeVersion") if check.status_code == 200 else None

            log_operation(
                "Workspace", ws_name, ws_name, "UpdateRuntime",
                "Succeeded" if observed == TARGET_RUNTIME else "Failed",
                f"Runtime observe = {observed}",
            )

        except Exception as ex:
            log_operation("Workspace", ws_name, ws_name, "UpdateRuntime", "Exception", str(ex)[:300])

        finally:
            if elevated and REVOKE_AFTER_MIGRATION:
                revoke(workspace_id, ws_name)

# ============================================================
# RAPPORT
# ============================================================

operations_df = pd.DataFrame(operations)
snapshots_df = pd.DataFrame(snapshots)

print("\n" + "=" * 70)
print("RESULTAT")
print("=" * 70)

if len(operations_df):
    print(operations_df["Status"].value_counts().to_string())

    failures = operations_df[~operations_df["Status"].isin(["Succeeded", "Manual"])]

    if len(failures):
        print(f"\n{len(failures)} operations en echec :")
        display(failures)

    display(operations_df)

else:
    print("Aucune operation d'ecriture executee.")

# Conserver les snapshots : ils sont la seule base de rollback.
# Runtime 1.2 etant hors support, un retour arriere n'est pas une strategie durable.
if len(snapshots_df):
    print(f"\n{len(snapshots_df)} snapshots de configuration conserves (rollback).")
